In [1]:
import pandas as pd
import polars as pl
import numpy as np
import os
import re
from collections import Counter
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from sklearn.preprocessing import MinMaxScaler


In [2]:
nltk.download('punkt')
nltk.download('stopwords')

stop_words = set(stopwords.words('english'))

[nltk_data] Downloading package punkt to /Users/isabel/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/isabel/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [3]:
INPUT_DIR = 'fromGoogleDrive'
OUTPUT_DIR = 'results'

In [4]:
os.makedirs(f'./{OUTPUT_DIR}/', exist_ok = True)

In [5]:
# preprocessing (same as compcor)
def preprocess(text):
    text = str(text).strip()
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"^\s*-\s*", "", text) # Remove dashes at the beginning of texts.
    text = re.sub(r"^\s*\d+\.\s*", "", text) # Remove numbers in 1., 2., 3. format at the beginning of the text.
    words = word_tokenize(text) # tokenize
    return words

In [6]:
metric_cols = [
    "Accuracy",
    "Weighted Accuracy",
    "Time",
    "Monotonicity",
    "Separability",
    "Linearity"
]

In [8]:
# Get dataset-specific information.
rows = []
for file in os.listdir(f'./{INPUT_DIR}/datasets/datasetsPrep/'):
    # if 'dementia' not in file:

        # Process texts.
        list_of_texts = [preprocess(t) for t in pd.read_csv(f'./{INPUT_DIR}/datasets/datasetsPrep/{file}')['text'].dropna().tolist()]
        # Flatten words.
        all_words = [word for doc in list_of_texts for word in doc]
        total_words = len(all_words)
        freq = Counter(all_words)
        # Filter stopwords.
        filtered_words = [w for w in all_words if w.lower() not in stop_words]
        filtered_word_total = len(filtered_words)
        stopword_ratio = (total_words - filtered_word_total) / total_words if total_words > 0 else 0
        # Average word length.
        avg_word_len = np.mean([len(w) for w in all_words]) if all_words else 0
        # Get document lengths.
        doc_lengths = np.array([len(t) for t in list_of_texts])
        # Vocab.
        vocab = set(all_words)
        vocab_len = len(vocab)
        # Type-Token Ratio.
        ttr = vocab_len / total_words if total_words > 0 else 0
        # Rare words (hapax legomena).
        hapax = sum(1 for _, c in freq.items() if c == 1)

        # Rare words (dis legomena).
        dis = sum(1 for _, c in freq.items() if c == 2)

        # Top-N Coverage (Frequency Concentration)
        top_10_count = sum(c for _, c in freq.most_common(10))
        coverage = top_10_count / total_words if total_words > 0 else 0
        # Append everything to a row. 
        rows.append({
        "Dataset": file.replace('.csv', ''),
        "Total Documents": len(list_of_texts),
        "Total Words": total_words,
        "Total Words (without stopwords)": filtered_word_total,
        "Total Stop Words": total_words - filtered_word_total,
        "Stopword Ratio": stopword_ratio,
        "Average Word Length": avg_word_len,
        "Vocab Length": vocab_len,
        "Type-Token Ratio": ttr,
        "Hapax Legomena": hapax,
        "Dis Legomena": dis,
        "Top 10 Words Coverage": coverage,
        "Mean Document Length": doc_lengths.mean(),
        "Median Document Length": np.median(doc_lengths),
        "Standard Deviation Document Length": doc_lengths.std(),
        "Min Document Length": doc_lengths.min(),
        "Max Document Length": doc_lengths.max(),
        "25th Document Length Percentile": np.percentile(doc_lengths, 25),
        "75th Document Length Percentile": np.percentile(doc_lengths, 75),
        })

# Make a dataframe.
dataset_characteristics_df = pd.DataFrame(rows)
# Remove non-comparable metrics (e.g. number of documents) for easy visualization. 
dataset_characteristics_df.drop(columns=[ 'Total Words', 'Total Words (without stopwords)', 'Total Stop Words', 'Vocab Length', 'Hapax Legomena', 'Dis Legomena', 'Standard Deviation Document Length', 'Min Document Length', 'Max Document Length', '25th Document Length Percentile', '75th Document Length Percentile'])

,Dataset,Total Documents,Stopword Ratio,Average Word Length,Type-Token Ratio,Top 10 Words Coverage,Mean Document Length,Median Document Length
0,yahoo,87362,0.424958,3.922350,0.029155,0.255595,47.847382,42.0
1,banking77,13069,0.498065,3.584884,0.017522,0.310065,13.249216,11.0
2,medicalAbstracts,14438,0.308262,5.109807,0.021201,0.263213,205.660064,200.0
3,dementiaAudio,549,0.453092,3.480664,0.026194,0.386441,116.338798,105.0
4,huffPostNews,189815,0.390420,4.199080,0.023466,0.238576,25.166567,23.0
5,clinc150,23700,0.498641,3.826311,0.036851,0.283618,8.525612,8.0
6,syntheticCareHomeNurseNotes,5783,0.340652,4.937623,0.027334,0.277734,26.532596,24.0
7,clinicalDialogueSummarizations,3603,0.398338,4.360843,0.035360,0.267681,46.278934,17.0
8,simSUM,10000,0.124586,4.106244,0.012563,0.381101,104.833600,103.0
9,atis,4978,0.425348,4.707788,0.015657,0.352524,11.367818,11.0


In [ ]:
def make_non_normalized_dfs(input_folder, output_file_name):
    all_temp_dfs = []
    for combination in os.listdir(f'./{INPUT_DIR}/outputCompcor/{input_folder}'):
        if os.path.isdir(f'./{INPUT_DIR}/outputCompcor/{input_folder}/{combination}'):
            combination_splits = combination.split('_')
            dataset1, dataset2, repetitions = combination_splits[0], combination_splits[1], combination_splits[3]
            temp_df = pd.read_csv(f'./{INPUT_DIR}/outputCompcor/{input_folder}/{combination}/{combination}_ksc_metrics_measures.csv')
            grouped_df = temp_df.groupby('metric')[metric_cols].mean()
            grouped_df['Dataset 1'] = dataset1
            grouped_df['Dataset 2'] = dataset2
            grouped_df['Repetitions'] = repetitions
            grouped_df['Metric'] = grouped_df.index
            grouped_df.reset_index(inplace=True)
            grouped_df.drop(columns='metric', inplace=True)
            all_temp_dfs.append(grouped_df)

    all_dfs = pd.concat(all_temp_dfs)

    print(# All datasets should have been compared the same number of times for this section to work.
    Counter(list(all_dfs['Dataset 1']) + list(all_dfs['Dataset 2'])))

    temp_dataset_dfs = []
    for dataset in dataset_characteristics_df['Dataset']:
        temp_df = all_dfs[
                (all_dfs['Dataset 1'] == dataset) | 
                (all_dfs['Dataset 2'] == dataset)
            ].copy()
        temp_df = temp_df.groupby('Metric')[metric_cols].mean()
        temp_df['Dataset'] = dataset
        temp_dataset_dfs.append(temp_df)
    dataset_df = pd.concat(temp_dataset_dfs)
    dataset_df.to_excel(f'./{OUTPUT_DIR}/{output_file_name}.xlsx')

    mean_dataset_df = dataset_df.groupby('Metric')[metric_cols].mean()
    mean_dataset_df.to_excel(f'./{OUTPUT_DIR}/{output_file_name}Mean.xlsx')

    return dataset_df, mean_dataset_df

In [ ]:
ksc_dataset_df, ksc_mean_dataset_df = make_non_normalized_dfs('ksc', 'kscDataset')
ksc_synth_dataset_df, ksc_synth_mean_dataset_df = make_non_normalized_dfs('ksc_synth', 'kscSynthDataset')

In [ ]:
ksc_dataset_df['Type'] = 'KSC'
ksc_synth_dataset_df['Type'] = 'KSC_Synth'

In [ ]:
all_dataset_df = pd.concat([ksc_dataset_df, ksc_synth_dataset_df]).groupby(['Dataset', 'Metric'])[metric_cols].mean()
all_mean_df = pd.concat([ksc_dataset_df, ksc_synth_dataset_df]).groupby(['Metric'])[metric_cols].mean()
all_type_mean_df = pd.concat([ksc_dataset_df, ksc_synth_dataset_df]).groupby(['Type', 'Metric'])[metric_cols].mean()
all_type_dataset_df = pd.concat([ksc_dataset_df, ksc_synth_dataset_df]).groupby(['Type', 'Dataset', 'Metric'])[metric_cols].mean()

In [ ]:
all_dataset_df.to_excel(f'./{OUTPUT_DIR}/allDataset.xlsx')
all_mean_df.to_excel(f'./{OUTPUT_DIR}/allDatasetMean.xlsx')

In [ ]:
all_type_mean_df

In [ ]:
all_type_dataset_df

In [ ]:
all_dataset_df

In [ ]:
all_mean_df